<a href="https://colab.research.google.com/github/shiqinxu777/ehrshot-benchmark/blob/main/Oncology_feasibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

ehr = pd.read_csv('EHR_Clinical_Data_10000.csv')
metadata = pd.read_csv('Radiology_Metadata_10000.csv')
reports = pd.read_csv('Radiology_Reports_10000.csv')

print("EHR shape:", ehr.shape)
print("Metadata shape:", metadata.shape)
print("Reports shape:", reports.shape)

EHR shape: (10000, 8)
Metadata shape: (10406, 7)
Reports shape: (9199, 2)


In [ ]:
print("=== EHR columns ===")
print(ehr.dtypes)
print()
print("=== Metadata columns ===")
print(metadata.dtypes)
print()
print("=== Reports columns ===")
print(reports.dtypes)

=== EHR columns ===
patient_id               object
dob                      object
sex                      object
diagnosis_code           object
diagnosis_description    object
diagnosis_date           object
encounter_type           object
problem_list_flag        object
dtype: object

=== Metadata columns ===
exam_id                object
patient_id             object
exam_date              object
modality               object
body_part              object
ordering_department    object
accession_number       object
dtype: object

=== Reports columns ===
exam_id        object
report_text    object
dtype: object


In [ ]:
print("=== EHR null counts ===")
print(ehr.isna().sum())
print()
print("=== Metadata null counts ===")
print(metadata.isna().sum())
print()
print("=== Reports null counts ===")
print(reports.isna().sum())

=== EHR null counts ===
patient_id                  0
dob                         0
sex                         0
diagnosis_code           1962
diagnosis_description       0
diagnosis_date            799
encounter_type              0
problem_list_flag           0
dtype: int64

=== Metadata null counts ===
exam_id                   0
patient_id                0
exam_date                 0
modality                  0
body_part              1682
ordering_department       0
accession_number          0
dtype: int64

=== Reports null counts ===
exam_id        0
report_text    0
dtype: int64


In [ ]:
print("EHR unique patients:", ehr['patient_id'].nunique())
print("Metadata unique patients:", metadata['patient_id'].nunique())
print("Metadata unique exams:", metadata['exam_id'].nunique())
print("Reports unique exams:", reports['exam_id'].nunique())


EHR unique patients: 10000
Metadata unique patients: 6550
Metadata unique exams: 10406
Reports unique exams: 9199


In [ ]:
metadata_exams = set(metadata['exam_id'])
reports_exams = set(reports['exam_id'])

in_metadata_not_reports = metadata_exams - reports_exams
in_reports_not_metadata = reports_exams - metadata_exams

print("Exams in metadata but no report:", len(in_metadata_not_reports))
print("Exams in reports but not in metadata:", len(in_reports_not_metadata))

Exams in metadata but no report: 1207
Exams in reports but not in metadata: 0


In [ ]:
# Count number of exams per patient to assess longitudinal depth

exams_per_patient = metadata.groupby('patient_id')['exam_id'].count()

print("Exams per patient:")
print(exams_per_patient.describe())
print()
print("Patients with only 1 exam:", (exams_per_patient == 1).sum())
print("Patients with 2+ exams:", (exams_per_patient >= 2).sum())
print("Patients with 3+ exams:", (exams_per_patient >= 3).sum())

Exams per patient:
count    6550.000000
mean        1.588702
std         0.735834
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         3.000000
Name: exam_id, dtype: float64

Patients with only 1 exam: 3674
Patients with 2+ exams: 2876
Patients with 3+ exams: 980


In [ ]:
# Examine modality and body_part distributions to understand imaging scope

print("=== Modality distribution ===")
print(metadata['modality'].value_counts())
print()
print("=== Body part distribution (top 15) ===")
print(metadata['body_part'].value_counts(dropna=False).head(15))

=== Modality distribution ===
modality
CT Abdomen/Pelvis              1344
Computed Tomography - Chest    1323
CT CHEST                       1320
MRI Brain                      1312
MRI Spine                      1283
CT Abdomen                     1276
MRI Pelvis                     1274
CT Chest W Contrast            1274
Name: count, dtype: int64

=== Body part distribution (top 15) ===
body_part
Chest      1798
Abdomen    1756
Pelvis     1742
Spine      1728
Brain      1700
NaN        1682
Name: count, dtype: int64


In [ ]:
# Examine ordering department distribution to assess oncology signal

print("=== Ordering department distribution ===")
print(metadata['ordering_department'].value_counts(dropna=False))

=== Ordering department distribution ===
ordering_department
ED           2240
Inpatient    2072
Urology      2070
Oncology     2031
Neurology    1993
Name: count, dtype: int64


In [ ]:
# Examine diagnosis code distribution, focus on oncology (ICD-10 C codes)

print("=== Total patients with any diagnosis code ===")
print(ehr['diagnosis_code'].notna().sum())
print()
print("=== Patients with C-code (oncology) diagnosis ===")
c_codes = ehr[ehr['diagnosis_code'].str.startswith('C', na=False)]
print(c_codes.shape[0])
print()
print("=== Top 15 C-code diagnoses ===")
print(c_codes['diagnosis_code'].value_counts().head(15))

=== Total patients with any diagnosis code ===
8038

=== Patients with C-code (oncology) diagnosis ===
4864

=== Top 15 C-code diagnoses ===
diagnosis_code
C34.90     316
C22.0      312
C15.9      309
C50.911    307
C43.9      305
C79.31     291
C54.1      290
C73        282
C61        281
C67.9      281
C56.9      281
C25.9      277
C71.9      273
C18.9      272
C64.9      272
Name: count, dtype: int64


In [ ]:
# Join oncology patients from EHR with metadata to find those with imaging

oncology_patients = ehr[ehr['diagnosis_code'].str.startswith('C', na=False)]['patient_id']

oncology_with_imaging = metadata[metadata['patient_id'].isin(oncology_patients)]

print("Oncology patients with C code:", len(oncology_patients))
print("Oncology patients who also have imaging:", oncology_with_imaging['patient_id'].nunique())
print("Total exams for these patients:", len(oncology_with_imaging))

Oncology patients with C code: 4864
Oncology patients who also have imaging: 3181
Total exams for these patients: 5131


In [ ]:
# Among oncology patients with imaging, assess longitudinal depth

oncology_exams_per_patient = oncology_with_imaging.groupby('patient_id')['exam_id'].count()

print("Exams per oncology patient:")
print(oncology_exams_per_patient.describe())
print()
print("Oncology patients with only 1 exam:", (oncology_exams_per_patient == 1).sum())
print("Oncology patients with 2+ exams:", (oncology_exams_per_patient >= 2).sum())
print("Oncology patients with 3+ exams:", (oncology_exams_per_patient >= 3).sum())

Exams per oncology patient:
count    3181.000000
mean        1.613015
std         0.741540
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         3.000000
Name: exam_id, dtype: float64

Oncology patients with only 1 exam: 1728
Oncology patients with 2+ exams: 1453
Oncology patients with 3+ exams: 497


In [ ]:
# Check report coverage for longitudinal oncology patients

longitudinal_oncology = oncology_exams_per_patient[oncology_exams_per_patient >= 2].index

longitudinal_exams = oncology_with_imaging[oncology_with_imaging['patient_id'].isin(longitudinal_oncology)]

longitudinal_with_reports = longitudinal_exams[longitudinal_exams['exam_id'].isin(reports['exam_id'])]

print("Total exams for longitudinal oncology patients:", len(longitudinal_exams))
print("Exams with reports:", len(longitudinal_with_reports))
print("Exams without reports:", len(longitudinal_exams) - len(longitudinal_with_reports))
print("Report coverage rate:", round(len(longitudinal_with_reports) / len(longitudinal_exams) * 100, 1), "%")

Total exams for longitudinal oncology patients: 3403
Exams with reports: 2977
Exams without reports: 426
Report coverage rate: 87.5 %


In [ ]:
# Sample report texts to understand content and structure

sample_reports = reports.sample(10, random_state=42)
for i, row in sample_reports.iterrows():
    print("=== Exam ID:", row['exam_id'], "===")
    print(row['report_text'][:500])
    print()

=== Exam ID: E007474 ===
Findings suspicious for malignancy.

=== Exam ID: E004850 ===
Mass concerning for carcinoma.

=== Exam ID: E002738 ===
Interval progression of metastatic disease.

=== Exam ID: E005338 ===
No acute abnormality.

=== Exam ID: E005360 ===
Stable treated lesion.

=== Exam ID: E005139 ===
Cannot exclude malignancy.

=== Exam ID: E000695 ===
Findings suspicious for malignancy.

=== Exam ID: E006990 ===
No evidence of recurrence.

=== Exam ID: E001506 ===
Stable treated lesion.

=== Exam ID: E003328 ===
Cannot exclude malignancy.



In [ ]:
# Check report text length distribution to assess data completeness

reports['text_length'] = reports['report_text'].str.len()

print("Report text length stats:")
print(reports['text_length'].describe())
print()
print("Reports under 100 characters:", (reports['text_length'] < 100).sum())
print("Reports under 50 characters:", (reports['text_length'] < 50).sum())

Report text length stats:
count    9199.000000
mean       29.045331
std         7.262213
min        21.000000
25%        22.000000
50%        26.000000
75%        35.000000
max        43.000000
Name: text_length, dtype: float64

Reports under 100 characters: 9199
Reports under 50 characters: 9199


In [ ]:
# Examine unique report text values to understand label taxonomy

print("Total unique report texts:", reports['report_text'].nunique())
print()
print("=== All unique report texts and their frequency ===")
print(reports['report_text'].value_counts())

Total unique report texts: 7

=== All unique report texts and their frequency ===
report_text
Interval progression of metastatic disease.    1349
Stable treated lesion.                         1341
Cannot exclude malignancy.                     1337
Findings suspicious for malignancy.            1308
No acute abnormality.                          1299
Mass concerning for carcinoma.                 1292
No evidence of recurrence.                     1273
Name: count, dtype: int64


In [ ]:
# Cell 16: Examine full diagnosis code distribution to understand what code types exist

print("=== All diagnosis code prefixes and counts ===")
ehr['code_prefix'] = ehr['diagnosis_code'].str[0]
print(ehr['code_prefix'].value_counts(dropna=False))

=== All diagnosis code prefixes and counts ===


NameError: name 'ehr' is not defined

In [ ]:
import pandas as pd

ehr = pd.read_csv('EHR_Clinical_Data_10000.csv')
metadata = pd.read_csv('Radiology_Metadata_10000.csv')
reports = pd.read_csv('Radiology_Reports_10000.csv')

print("EHR shape:", ehr.shape)
print("Metadata shape:", metadata.shape)
print("Reports shape:", reports.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'EHR_Clinical_Data_10000.csv'

In [ ]:
import pandas as pd

ehr = pd.read_csv('EHR_Clinical_Data_10000.csv')
metadata = pd.read_csv('Radiology_Metadata_10000.csv')
reports = pd.read_csv('Radiology_Reports_10000.csv')

print("EHR shape:", ehr.shape)
print("Metadata shape:", metadata.shape)
print("Reports shape:", reports.shape)

EHR shape: (10000, 8)
Metadata shape: (10406, 7)
Reports shape: (9199, 2)


In [ ]:
# Cell 16: Examine full diagnosis code distribution to understand what code types exist

print("=== All diagnosis code prefixes and counts ===")
ehr['code_prefix'] = ehr['diagnosis_code'].str[0]
print(ehr['code_prefix'].value_counts(dropna=False))

=== All diagnosis code prefixes and counts ===
code_prefix
C      4864
D      2249
NaN    1962
Z       925
Name: count, dtype: int64


In [ ]:
# Cell 17: Examine D codes and Z codes in detail to inform include/exclude decisions

print("=== D code distribution (top 15) ===")
d_codes = ehr[ehr['diagnosis_code'].str.startswith('D', na=False)]
print(d_codes['diagnosis_code'].value_counts().head(15))
print()
print("=== Z code distribution ===")
z_codes = ehr[ehr['diagnosis_code'].str.startswith('Z', na=False)]
print(z_codes['diagnosis_code'].value_counts())

=== D code distribution (top 15) ===
diagnosis_code
D24.1    517
D12.6    499
D13.4    491
D05.1    399
D09.0    343
Name: count, dtype: int64

=== Z code distribution ===
diagnosis_code
Z85.038    321
Z85.3      315
Z85.118    289
Name: count, dtype: int64


In [ ]:
# Cell 18: Apply cohort policy — define primary, sensitivity, and exploratory cohorts

# Primary cohort: C codes only
primary_cohort = ehr[ehr['diagnosis_code'].str.startswith('C', na=False)]

# Sensitivity cohort: C codes + D05 and D09
sensitivity_cohort = ehr[
    ehr['diagnosis_code'].str.startswith('C', na=False) |
    ehr['diagnosis_code'].str.startswith('D05', na=False) |
    ehr['diagnosis_code'].str.startswith('D09', na=False)
]

# Exploratory cohort: C + D05 + D09 + Z85
exploratory_cohort = ehr[
    ehr['diagnosis_code'].str.startswith('C', na=False) |
    ehr['diagnosis_code'].str.startswith('D05', na=False) |
    ehr['diagnosis_code'].str.startswith('D09', na=False) |
    ehr['diagnosis_code'].str.startswith('Z85', na=False)
]

print("Primary cohort (C codes):", len(primary_cohort))
print("Sensitivity cohort (C + D05 + D09):", len(sensitivity_cohort))
print("Exploratory cohort (C + D05 + D09 + Z85):", len(exploratory_cohort))

Primary cohort (C codes): 4864
Sensitivity cohort (C + D05 + D09): 5606
Exploratory cohort (C + D05 + D09 + Z85): 6531


In [ ]:
# Cell 19: Join primary cohort with metadata, then tier by ordering department

primary_with_imaging = metadata[metadata['patient_id'].isin(primary_cohort['patient_id'])]

print("Primary cohort patients with imaging:", primary_with_imaging['patient_id'].nunique())
print("Total exams:", len(primary_with_imaging))
print()
print("=== Exam count by ordering department ===")
print(primary_with_imaging['ordering_department'].value_counts())

Primary cohort patients with imaging: 3181
Total exams: 5131

=== Exam count by ordering department ===
ordering_department
Urology      1068
ED           1057
Inpatient    1045
Oncology      982
Neurology     979
Name: count, dtype: int64


In [ ]:
# Cell 20: Apply ordering department confidence tiering

# High-confidence tier: Oncology-ordered
high_confidence = primary_with_imaging[
    primary_with_imaging['ordering_department'] == 'Oncology'
]

# Supporting tier: non-Oncology but with confirmed C code (already filtered)
supporting_tier = primary_with_imaging[
    primary_with_imaging['ordering_department'] != 'Oncology'
]

print("High-confidence tier (Oncology-ordered):", len(high_confidence))
print("Unique patients:", high_confidence['patient_id'].nunique())
print()
print("Supporting tier (non-Oncology, confirmed C code):", len(supporting_tier))
print("Unique patients:", supporting_tier['patient_id'].nunique())

High-confidence tier (Oncology-ordered): 982
Unique patients: 893

Supporting tier (non-Oncology, confirmed C code): 4149
Unique patients: 2808


In [ ]:
# Cell 21: Identify patients whose ALL exams are Oncology-ordered vs. mixed

patient_dept = primary_with_imaging.groupby('patient_id')['ordering_department'].nunique()

oncology_only_patients = primary_with_imaging[
    primary_with_imaging['patient_id'].isin(patient_dept[patient_dept == 1].index) &
    (primary_with_imaging['ordering_department'] == 'Oncology')
]['patient_id'].nunique()

print("Patients with ONLY Oncology-ordered exams:", oncology_only_patients)
print("Patients with mixed department exams:", (patient_dept > 1).sum())
print("Patients with only non-Oncology exams:", (patient_dept == 1).sum() - oncology_only_patients)

Patients with ONLY Oncology-ordered exams: 373
Patients with mixed department exams: 1230
Patients with only non-Oncology exams: 1578


In [ ]:
# Cell 22: Calculate exam counts for all confidence tiers

# Tier 1a: patients whose ALL exams are Oncology-ordered
oncology_only_patient_ids = patient_dept[patient_dept == 1].index
tier_1a_patients = primary_with_imaging[
    primary_with_imaging['patient_id'].isin(oncology_only_patient_ids) &
    (primary_with_imaging['ordering_department'] == 'Oncology')
]

# Tier 2: patients with ONLY non-Oncology exams
non_oncology_only_patient_ids = primary_with_imaging[
    ~primary_with_imaging['patient_id'].isin(
        primary_with_imaging[primary_with_imaging['ordering_department'] == 'Oncology']['patient_id']
    )
]['patient_id'].unique()
tier_2 = primary_with_imaging[primary_with_imaging['patient_id'].isin(non_oncology_only_patient_ids)]

print("Tier 1a — all exams Oncology-ordered:")
print("  Patients:", tier_1a_patients['patient_id'].nunique())
print("  Exams:", len(tier_1a_patients))
print()
print("Tier 2 — non-Oncology only:")
print("  Patients:", len(non_oncology_only_patient_ids))
print("  Exams:", len(tier_2))

Tier 1a — all exams Oncology-ordered:
  Patients: 373
  Exams: 414

Tier 2 — non-Oncology only:
  Patients: 2288
  Exams: 3447


In [ ]:
# Cell 23: Examine diagnosis descriptions for patients with null diagnosis codes

null_code_patients = ehr[ehr['diagnosis_code'].isna()]
print("Patients with null diagnosis code:", len(null_code_patients))
print()
print("=== Diagnosis descriptions for null-code patients ===")
print(null_code_patients['diagnosis_description'].value_counts().head(20))

Patients with null diagnosis code: 1962

=== Diagnosis descriptions for null-code patients ===
diagnosis_description
Routine visit                             786
Cannot exclude malignancy                 322
Rule out lung cancer                      288
Suspicious mass                           288
Pulmonary nodule suspicious for cancer    278
Name: count, dtype: int64


In [1]:
import pandas as pd

ehr = pd.read_csv('EHR_Clinical_Data_10000.csv')
metadata = pd.read_csv('Radiology_Metadata_10000.csv')
reports = pd.read_csv('Radiology_Reports_10000.csv')

print("EHR shape:", ehr.shape)
print("Metadata shape:", metadata.shape)
print("Reports shape:", reports.shape)

EHR shape: (10000, 8)
Metadata shape: (10406, 7)
Reports shape: (9199, 2)


In [2]:
# Cell 24: Examine raw modality values to plan standardization

print("=== Raw modality values ===")
print(metadata['modality'].value_counts())

=== Raw modality values ===
modality
CT Abdomen/Pelvis              1344
Computed Tomography - Chest    1323
CT CHEST                       1320
MRI Brain                      1312
MRI Spine                      1283
CT Abdomen                     1276
MRI Pelvis                     1274
CT Chest W Contrast            1274
Name: count, dtype: int64


In [3]:
# Cell 25: Check inconsistency between modality and body_part fields

# Define expected body_part from modality
def extract_body_from_modality(modality):
    modality = modality.upper()
    if 'CHEST' in modality:
        return 'Chest'
    elif 'ABDOMEN' in modality:
        return 'Abdomen'
    elif 'PELVIS' in modality:
        return 'Pelvis'
    elif 'BRAIN' in modality:
        return 'Brain'
    elif 'SPINE' in modality:
        return 'Spine'
    else:
        return None

metadata['body_from_modality'] = metadata['modality'].apply(extract_body_from_modality)

# Find rows where both fields are non-null but disagree
mismatch = metadata[
    metadata['body_part'].notna() &
    metadata['body_from_modality'].notna() &
    (metadata['body_part'] != metadata['body_from_modality'])
]

print("Total exams with body_part:", metadata['body_part'].notna().sum())
print("Total exams with extractable body from modality:", metadata['body_from_modality'].notna().sum())
print("Mismatches between body_part and modality:", len(mismatch))
print()
print("=== Sample mismatches ===")
print(mismatch[['exam_id', 'modality', 'body_part']].head(10))

Total exams with body_part: 8724
Total exams with extractable body from modality: 10406
Mismatches between body_part and modality: 6913

=== Sample mismatches ===
    exam_id                     modality body_part
0   E000001  Computed Tomography - Chest   Abdomen
1   E000002  Computed Tomography - Chest     Brain
3   E000004                   MRI Pelvis     Chest
5   E000006            CT Abdomen/Pelvis     Chest
6   E000007            CT Abdomen/Pelvis     Chest
7   E000008  Computed Tomography - Chest   Abdomen
10  E000011            CT Abdomen/Pelvis     Spine
11  E000012                     CT CHEST   Abdomen
12  E000013            CT Abdomen/Pelvis     Spine
13  E000014                    MRI Spine     Chest


In [4]:
# Cell 26: Analyze mismatch patterns between modality-derived body part and body_part field

mismatch['modality_clean'] = mismatch['modality'].apply(standardize_modality)

print("=== Top mismatch pairs (modality body → body_part) ===")
mismatch_pairs = mismatch.groupby(['body_from_modality', 'body_part']).size().reset_index(name='count')
mismatch_pairs = mismatch_pairs.sort_values('count', ascending=False)
print(mismatch_pairs.head(20))
print()
print("=== Mismatch rate by modality (CT vs MRI) ===")
for mod in ['CT', 'MRI']:
    mod_total = metadata[metadata['modality_clean'] == mod]['body_part'].notna().sum()
    mod_mismatch = mismatch[mismatch['modality_clean'] == mod].shape[0]
    print(f"{mod}: {mod_mismatch} mismatches out of {mod_total} ({round(mod_mismatch/mod_total*100,1)}%)")

NameError: name 'standardize_modality' is not defined

In [5]:
# Re-define functions after session reset

def standardize_modality(value):
    if 'CT' in value or 'Computed Tomography' in value:
        return 'CT'
    elif 'MRI' in value:
        return 'MRI'
    else:
        return 'Other'

def extract_body_from_modality(modality):
    modality = modality.upper()
    if 'CHEST' in modality:
        return 'Chest'
    elif 'ABDOMEN' in modality:
        return 'Abdomen'
    elif 'PELVIS' in modality:
        return 'Pelvis'
    elif 'BRAIN' in modality:
        return 'Brain'
    elif 'SPINE' in modality:
        return 'Spine'
    else:
        return None

metadata['modality_clean'] = metadata['modality'].apply(standardize_modality)
metadata['body_from_modality'] = metadata['modality'].apply(extract_body_from_modality)

mismatch = metadata[
    metadata['body_part'].notna() &
    metadata['body_from_modality'].notna() &
    (metadata['body_part'] != metadata['body_from_modality'])
]

print("Functions defined, mismatch count:", len(mismatch))

Functions defined, mismatch count: 6913


In [6]:
# Cell 26: Analyze mismatch patterns between modality-derived body part and body_part field

mismatch['modality_clean'] = mismatch['modality'].apply(standardize_modality)

print("=== Top mismatch pairs (modality body → body_part) ===")
mismatch_pairs = mismatch.groupby(['body_from_modality', 'body_part']).size().reset_index(name='count')
mismatch_pairs = mismatch_pairs.sort_values('count', ascending=False)
print(mismatch_pairs.head(20))
print()
print("=== Mismatch rate by modality (CT vs MRI) ===")
for mod in ['CT', 'MRI']:
    mod_total = metadata[metadata['modality_clean'] == mod]['body_part'].notna().sum()
    mod_mismatch = mismatch[mismatch['modality_clean'] == mod].shape[0]
    print(f"{mod}: {mod_mismatch} mismatches out of {mod_total} ({round(mod_mismatch/mod_total*100,1)}%)")

=== Top mismatch pairs (modality body → body_part) ===
   body_from_modality body_part  count
11              Chest     Spine    676
8               Chest   Abdomen    652
9               Chest     Brain    645
10              Chest    Pelvis    617
2             Abdomen    Pelvis    457
0             Abdomen     Brain    443
1             Abdomen     Chest    425
3             Abdomen     Spine    403
19              Spine    Pelvis    243
5               Brain     Chest    242
14             Pelvis     Chest    225
7               Brain     Spine    222
15             Pelvis     Spine    218
16              Spine   Abdomen    217
18              Spine     Chest    216
4               Brain   Abdomen    216
12             Pelvis   Abdomen    207
6               Brain    Pelvis    204
17              Spine     Brain    197
13             Pelvis     Brain    188

=== Mismatch rate by modality (CT vs MRI) ===
CT: 4318 mismatches out of 5472 (78.9%)
MRI: 2595 mismatches out of 3252 (79.8%

/tmp/ipykernel_607/4041044320.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mismatch['modality_clean'] = mismatch['modality'].apply(standardize_modality)


In [7]:
# Cell 27: Sample mismatch cases where modality=Chest but body_part=Brain, check report text

chest_brain_mismatch = mismatch[
    (mismatch['body_from_modality'] == 'Chest') &
    (mismatch['body_part'] == 'Brain')
]

# Join with reports
chest_brain_with_reports = chest_brain_mismatch.merge(reports, on='exam_id', how='inner')

print("Chest modality / Brain body_part cases with reports:", len(chest_brain_with_reports))
print()
print("=== Sample 20 report texts ===")
sample = chest_brain_with_reports.sample(min(20, len(chest_brain_with_reports)), random_state=42)
for i, row in sample.iterrows():
    print(f"Exam: {row['exam_id']} | Dept: {row['ordering_department']}")
    print(f"Report: {row['report_text']}")
    print()

Chest modality / Brain body_part cases with reports: 563

=== Sample 20 report texts ===
Exam: E004668 | Dept: Urology
Report: Cannot exclude malignancy.

Exam: E009754 | Dept: Neurology
Report: Cannot exclude malignancy.

Exam: E005105 | Dept: Inpatient
Report: Interval progression of metastatic disease.

Exam: E000950 | Dept: Urology
Report: Mass concerning for carcinoma.

Exam: E010292 | Dept: ED
Report: Cannot exclude malignancy.

Exam: E006042 | Dept: Oncology
Report: Stable treated lesion.

Exam: E004580 | Dept: Neurology
Report: Stable treated lesion.

Exam: E009747 | Dept: Oncology
Report: Cannot exclude malignancy.

Exam: E001319 | Dept: ED
Report: Interval progression of metastatic disease.

Exam: E009957 | Dept: Urology
Report: Findings suspicious for malignancy.

Exam: E004920 | Dept: Neurology
Report: Stable treated lesion.

Exam: E001342 | Dept: Oncology
Report: Mass concerning for carcinoma.

Exam: E004381 | Dept: Urology
Report: No acute abnormality.

Exam: E007785 | De

In [8]:
# Cell 28: Rebuild primary cohort and join with metadata and reports

# Primary cohort: C codes only
primary_cohort = ehr[ehr['diagnosis_code'].str.startswith('C', na=False)]

# Join with metadata
primary_with_imaging = metadata[metadata['patient_id'].isin(primary_cohort['patient_id'])].copy()

# Add modality_clean
primary_with_imaging['modality_clean'] = primary_with_imaging['modality'].apply(standardize_modality)

# Add report linkage flag
primary_with_imaging['has_report'] = primary_with_imaging['exam_id'].isin(reports['exam_id'])

print("Primary cohort patients:", primary_cohort['patient_id'].nunique())
print("Primary cohort exams:", len(primary_with_imaging))
print("Exams with linked report:", primary_with_imaging['has_report'].sum())
print("Report linkage rate:", round(primary_with_imaging['has_report'].mean() * 100, 1), "%")

Primary cohort patients: 4864
Primary cohort exams: 5131
Exams with linked report: 4507
Report linkage rate: 87.8 %


In [9]:
# Cell 29: Examine modality distribution within primary cohort exams

print("=== Modality distribution in primary cohort ===")
print(primary_with_imaging['modality_clean'].value_counts())
print()
print("=== Modality distribution with report linkage rate ===")
for mod in ['CT', 'MRI', 'Other']:
    mod_exams = primary_with_imaging[primary_with_imaging['modality_clean'] == mod]
    total = len(mod_exams)
    with_report = mod_exams['has_report'].sum()
    print(f"{mod}: {total} exams, {with_report} with report ({round(with_report/total*100,1)}%)")

=== Modality distribution in primary cohort ===
modality_clean
CT     3246
MRI    1885
Name: count, dtype: int64

=== Modality distribution with report linkage rate ===
CT: 3246 exams, 2852 with report (87.9%)
MRI: 1885 exams, 1655 with report (87.8%)
Other: 0 exams, 0 with report (nan%)


/tmp/ipykernel_607/1497075303.py:11: RuntimeWarning: invalid value encountered in scalar divide
  print(f"{mod}: {total} exams, {with_report} with report ({round(with_report/total*100,1)}%)")


In [10]:
# Cell 30: Identify longitudinal pairs by modality type within primary cohort

# Get exams per patient with modality info
patient_modality = primary_with_imaging.groupby('patient_id')['modality_clean'].apply(list).reset_index()
patient_modality.columns = ['patient_id', 'modalities']

def classify_modality_pattern(modalities):
    unique_mods = set(modalities)
    if len(modalities) == 1:
        return 'single_exam'
    elif unique_mods == {'CT'}:
        return 'CT_only'
    elif unique_mods == {'MRI'}:
        return 'MRI_only'
    else:
        return 'mixed_modality'

patient_modality['modality_pattern'] = patient_modality['modalities'].apply(classify_modality_pattern)

print("=== Patient modality patterns ===")
print(patient_modality['modality_pattern'].value_counts())

=== Patient modality patterns ===
modality_pattern
single_exam       1728
mixed_modality     816
CT_only            492
MRI_only           145
Name: count, dtype: int64


In [11]:
# Cell 31: Check report completeness for CT-only longitudinal patients

ct_only_patients = patient_modality[patient_modality['modality_pattern'] == 'CT_only']['patient_id']
ct_only_exams = primary_with_imaging[primary_with_imaging['patient_id'].isin(ct_only_patients)]

# Check report linkage
ct_only_complete = ct_only_exams.groupby('patient_id')['has_report'].all()

print("CT-only longitudinal patients:", len(ct_only_patients))
print("Total exams:", len(ct_only_exams))
print()
print("Patients with ALL exams linked to report:", ct_only_complete.sum())
print("Patients with at least one exam missing report:", (~ct_only_complete).sum())

CT-only longitudinal patients: 492
Total exams: 1102

Patients with ALL exams linked to report: 356
Patients with at least one exam missing report: 136


In [12]:
# Cell 32: Build complete patient funnel summary

print("=== Complete Patient Funnel ===")
print(f"Total EHR patients:                          10,000")
print(f"Patients with C code (primary cohort):        4,864")
print(f"Primary cohort with any imaging:              3,181")
print(f"  - Single exam only:                         1,728")
print(f"  - Longitudinal (2+ exams):                  1,453")
print(f"    - CT-only longitudinal:                     492")
print(f"      - All exams have report:                  356")
print(f"      - At least one exam missing report:       136")
print(f"    - MRI-only longitudinal:                    145")
print(f"    - Mixed-modality longitudinal:              816")

=== Complete Patient Funnel ===
Total EHR patients:                          10,000
Patients with C code (primary cohort):        4,864
Primary cohort with any imaging:              3,181
  - Single exam only:                         1,728
  - Longitudinal (2+ exams):                  1,453
    - CT-only longitudinal:                     492
      - All exams have report:                  356
      - At least one exam missing report:       136
    - MRI-only longitudinal:                    145
    - Mixed-modality longitudinal:              816


In [13]:
# Cell 33: Calculate Tier 1 — CT-only longitudinal + all reports + at least one Oncology-ordered exam

# Get CT-only longitudinal patients with complete report coverage
ct_only_complete_patients = ct_only_complete[ct_only_complete].index

# Among these, find those with at least one Oncology-ordered exam
ct_only_complete_exams = primary_with_imaging[
    primary_with_imaging['patient_id'].isin(ct_only_complete_patients)
]

tier_1_patients = ct_only_complete_exams[
    ct_only_complete_exams['ordering_department'] == 'Oncology'
]['patient_id'].nunique()

print("Tier 1 patients (CT-only + all reports + any Oncology-ordered exam):", tier_1_patients)

Tier 1 patients (CT-only + all reports + any Oncology-ordered exam): 108


In [14]:
# Cell 34: Verify all expansion path patient counts

# Base case: CT-only longitudinal + all exams have report
base_case = ct_only_complete[ct_only_complete].index
print("Base case (CT-only + complete report):", len(base_case))

# Accept partial report coverage: all CT-only longitudinal
ct_only_all = patient_modality[patient_modality['modality_pattern'] == 'CT_only']['patient_id']
print("Accept partial report coverage (CT-only longitudinal):", len(ct_only_all))

# Accept MRI-only longitudinal: base case + MRI-only
mri_only = patient_modality[patient_modality['modality_pattern'] == 'MRI_only']['patient_id']
base_plus_mri = len(base_case) + len(mri_only)
print("Accept MRI-only longitudinal (base + MRI-only):", base_plus_mri)

# Accept mixed modality: all longitudinal patients
longitudinal_all = patient_modality[patient_modality['modality_pattern'] != 'single_exam']['patient_id']
print("Accept mixed modality (all longitudinal):", len(longitudinal_all))

# Accept single-exam: full primary cohort with imaging
print("Accept single-exam patients (full primary cohort with imaging):", primary_with_imaging['patient_id'].nunique())

Base case (CT-only + complete report): 356
Accept partial report coverage (CT-only longitudinal): 492
Accept MRI-only longitudinal (base + MRI-only): 501
Accept mixed modality (all longitudinal): 1453
Accept single-exam patients (full primary cohort with imaging): 3181


In [15]:
# Cell 35: Convert exam_date to datetime and calculate longitudinal span for CT-only patients

# Convert exam_date to datetime
metadata['exam_date'] = pd.to_datetime(metadata['exam_date'])

# Get CT-only longitudinal patients
ct_only_patient_ids = patient_modality[patient_modality['modality_pattern'] == 'CT_only']['patient_id']
ct_only_exams = primary_with_imaging[primary_with_imaging['patient_id'].isin(ct_only_patient_ids)].copy()

# Calculate span per patient
ct_span = ct_only_exams.groupby('patient_id')['exam_date'].agg(['min', 'max'])
ct_span['span_days'] = (ct_span['max'] - ct_span['min']).dt.days

print("=== CT-only longitudinal span summary (days) ===")
print(ct_span['span_days'].describe())
print()
print("Patients with span > 180 days (6 months):", (ct_span['span_days'] > 180).sum())
print("Patients with span > 365 days (1 year):", (ct_span['span_days'] > 365).sum())
print()
print("% with span > 6 months:", round((ct_span['span_days'] > 180).mean() * 100, 1), "%")
print("% with span > 1 year:", round((ct_span['span_days'] > 365).mean() * 100, 1), "%")

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [16]:
# Cell 35: Convert exam_date to datetime and calculate longitudinal span for CT-only patients

# Convert exam_date to datetime
metadata['exam_date'] = pd.to_datetime(metadata['exam_date'])

# Get CT-only longitudinal patients
ct_only_patient_ids = patient_modality[patient_modality['modality_pattern'] == 'CT_only']['patient_id']
ct_only_exams = primary_with_imaging[primary_with_imaging['patient_id'].isin(ct_only_patient_ids)].copy()

# Calculate span per patient
ct_span = ct_only_exams.groupby('patient_id')['exam_date'].agg(['min', 'max'])
ct_span['span_days'] = (ct_span['max'] - ct_span['min']).dt.days

print("=== CT-only longitudinal span summary (days) ===")
print(ct_span['span_days'].describe())
print()
print("Patients with span > 180 days (6 months):", (ct_span['span_days'] > 180).sum())
print("Patients with span > 365 days (1 year):", (ct_span['span_days'] > 365).sum())
print()
print("% with span > 6 months:", round((ct_span['span_days'] > 180).mean() * 100, 1), "%")
print("% with span > 1 year:", round((ct_span['span_days'] > 365).mean() * 100, 1), "%")

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [17]:
# Re-convert exam_date to datetime after session reset

metadata['exam_date'] = pd.to_datetime(metadata['exam_date'])
print("exam_date dtype:", metadata['exam_date'].dtype)
print("Sample dates:", metadata['exam_date'].head(3).tolist())

exam_date dtype: datetime64[ns]
Sample dates: [Timestamp('2023-02-18 00:00:00'), Timestamp('2024-09-14 00:00:00'), Timestamp('2022-04-23 00:00:00')]


In [18]:
# Cell 35: Convert exam_date to datetime and calculate longitudinal span for CT-only patients

# Get CT-only longitudinal patients
ct_only_patient_ids = patient_modality[patient_modality['modality_pattern'] == 'CT_only']['patient_id']
ct_only_exams = primary_with_imaging[primary_with_imaging['patient_id'].isin(ct_only_patient_ids)].copy()

# Make sure exam_date is datetime in this subset
ct_only_exams['exam_date'] = pd.to_datetime(ct_only_exams['exam_date'])

# Calculate span per patient
ct_span = ct_only_exams.groupby('patient_id')['exam_date'].agg(['min', 'max'])
ct_span['span_days'] = (ct_span['max'] - ct_span['min']).dt.days

print("=== CT-only longitudinal span summary (days) ===")
print(ct_span['span_days'].describe())
print()
print("Patients with span > 180 days (6 months):", (ct_span['span_days'] > 180).sum())
print("Patients with span > 365 days (1 year):", (ct_span['span_days'] > 365).sum())
print()
print("% with span > 6 months:", round((ct_span['span_days'] > 180).mean() * 100, 1), "%")
print("% with span > 1 year:", round((ct_span['span_days'] > 365).mean() * 100, 1), "%")

=== CT-only longitudinal span summary (days) ===
count     492.000000
mean     1108.743902
std       706.955297
min        11.000000
25%       525.750000
50%      1069.500000
75%      1624.750000
max      2848.000000
Name: span_days, dtype: float64

Patients with span > 180 days (6 months): 443
Patients with span > 365 days (1 year): 397

% with span > 6 months: 90.0 %
% with span > 1 year: 80.7 %


In [19]:
# Cell 36: Compare span for all longitudinal patients vs CT-only

# All longitudinal patients
all_longitudinal_ids = patient_modality[
    patient_modality['modality_pattern'] != 'single_exam'
]['patient_id']
all_long_exams = primary_with_imaging[
    primary_with_imaging['patient_id'].isin(all_longitudinal_ids)
].copy()
all_long_exams['exam_date'] = pd.to_datetime(all_long_exams['exam_date'])

all_span = all_long_exams.groupby('patient_id')['exam_date'].agg(['min', 'max'])
all_span['span_days'] = (all_span['max'] - all_span['min']).dt.days

print("=== All longitudinal patients span summary (days) ===")
print(all_span['span_days'].describe())
print()
print("% with span > 6 months:", round((all_span['span_days'] > 180).mean() * 100, 1), "%")
print("% with span > 1 year:", round((all_span['span_days'] > 365).mean() * 100, 1), "%")

=== All longitudinal patients span summary (days) ===
count    1453.000000
mean     1142.128011
std       718.604313
min         1.000000
25%       539.000000
50%      1085.000000
75%      1670.000000
max      2862.000000
Name: span_days, dtype: float64

% with span > 6 months: 91.9 %
% with span > 1 year: 82.9 %


In [20]:
# Cell 37: Map report labels to three-bucket structure and count by bucket

# Define bucket mapping
bucket_map = {
    'Interval progression of metastatic disease.': 'Progression',
    'Stable treated lesion.': 'Stable or improving',
    'No evidence of recurrence.': 'Stable or improving',
    'No acute abnormality.': 'Stable or improving',
    'Cannot exclude malignancy.': 'Indeterminate',
    'Findings suspicious for malignancy.': 'Indeterminate',
    'Mass concerning for carcinoma.': 'Indeterminate'
}

# Get CT-only longitudinal exams with reports
ct_only_long_with_reports = ct_only_exams[ct_only_exams['has_report']].merge(
    reports, on='exam_id', how='inner'
)

# Map to buckets
ct_only_long_with_reports['label_bucket'] = ct_only_long_with_reports['report_text'].map(bucket_map)

print("=== Exam count by bucket (CT-only longitudinal) ===")
print(ct_only_long_with_reports['label_bucket'].value_counts())
print()
print("=== Patient count by bucket ===")
print(ct_only_long_with_reports.groupby('label_bucket')['patient_id'].nunique())

=== Exam count by bucket (CT-only longitudinal) ===
label_bucket
Indeterminate          432
Stable or improving    381
Progression            137
Name: count, dtype: int64

=== Patient count by bucket ===
label_bucket
Indeterminate          337
Progression            124
Stable or improving    294
Name: patient_id, dtype: int64


In [21]:
# Cell 38: Assess how many patients have at least one progression label
# vs. patients who are consistently stable or indeterminate

patient_buckets = ct_only_long_with_reports.groupby('patient_id')['label_bucket'].apply(set)

has_progression = patient_buckets.apply(lambda x: 'Progression' in x)
has_stable = patient_buckets.apply(lambda x: 'Stable or improving' in x)
has_indeterminate = patient_buckets.apply(lambda x: 'Indeterminate' in x)
only_indeterminate = patient_buckets.apply(lambda x: x == {'Indeterminate'})
mixed = patient_buckets.apply(lambda x: len(x) > 1)

print("Patients with at least one Progression label:", has_progression.sum())
print("Patients with at least one Stable or improving label:", has_stable.sum())
print("Patients with at least one Indeterminate label:", has_indeterminate.sum())
print()
print("Patients with ONLY Indeterminate labels:", only_indeterminate.sum())
print("Patients with mixed buckets across exams:", mixed.sum())

Patients with at least one Progression label: 124
Patients with at least one Stable or improving label: 294
Patients with at least one Indeterminate label: 337

Patients with ONLY Indeterminate labels: 120
Patients with mixed buckets across exams: 257


In [22]:
# Cell 39: Identify patients with progression signal following stable/improving label
# This is the ideal longitudinal pattern: stable → progression

def has_progression_after_stable(patient_id):
    patient_exams = ct_only_long_with_reports[
        ct_only_long_with_reports['patient_id'] == patient_id
    ].sort_values('exam_date')

    buckets = patient_exams['label_bucket'].tolist()

    # Check if Stable or improving appears before Progression
    for i in range(len(buckets) - 1):
        if buckets[i] == 'Stable or improving' and 'Progression' in buckets[i+1:]:
            return True
    return False

# Apply to patients with both progression and stable labels
both_labels = patient_buckets[
    patient_buckets.apply(lambda x: 'Progression' in x and 'Stable or improving' in x)
].index

stable_then_progression = sum(has_progression_after_stable(p) for p in both_labels)

print("Patients with both Progression and Stable labels:", len(both_labels))
print("Patients with Stable → Progression pattern:", stable_then_progression)

Patients with both Progression and Stable labels: 54
Patients with Stable → Progression pattern: 26
